# Chart Pattern Detector - Image Testing Notebook
Use this notebook to test the detector on screenshots (e.g. from Investing.com).

In [1]:
import sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import cv2
import warnings
import os

# Suppress sklearn version warnings
warnings.filterwarnings("ignore")

# Ensure plots show in the notebook
%matplotlib inline

# Add project root to path so we can import 'scripts'
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

# Import the inference function
from scripts.predict import predict_image_process

In [2]:
# Setup paths
images_dir = Path("images_for_testing")
out_dir = Path("../reports/infer/notebook_test")
pipeline_cfg = Path("../configs/pipeline.yaml")
models_root = Path("../reports/baselines")

# Clean up previous run CSV to ensure fresh schema
csv_out = out_dir / "predictions.csv"
if csv_out.exists():
    try:
        os.remove(csv_out)
        print("🧹 Cleared previous predictions.csv")
    except OSError:
        pass

# Create images_dir if it doesn't exist
images_dir.mkdir(exist_ok=True)

# Check for images
images = list(images_dir.glob("*"))
valid_exts = {'.png', '.jpg', '.jpeg', '.webp'}
images = [i for i in images if i.suffix.lower() in valid_exts]

if not images:
    print(f"⚠️  No images found in {images_dir.absolute()}")
    print("Please copy your screenshots (png/jpg) into this folder and re-run this cell.")
else:
    print(f"✅ Found {len(images)} images: {[f.name for f in images]}")

🧹 Cleared previous predictions.csv
✅ Found 5 images: ['Screenshot from 2025-12-20 15-56-24.png', 'Screenshot from 2025-12-20 15-58-16.png', 'Screenshot from 2025-12-20 15-57-55.png', 'Screenshot from 2025-12-20 15-57-43.png', 'Screenshot from 2025-12-20 15-57-19.png']


In [3]:
# Run Inference Loop
print("Starting Inference... (Plots will appear below)\n")

for img_path in images:
    try:
        # Call the inference function directly
        # This bypasses OHLCV loading and uses pure CV
        csv_path = predict_image_process(
            image_path=img_path,
            out_dir=out_dir,
            pipeline_cfg_path=pipeline_cfg,
            models_root=models_root
        )
        
        # Read results
        df = pd.read_csv(csv_path)
        # The CSV might contain multiple rows now (append mode). 
        # We want the LAST row corresponding to this image.
        # But wait, we just want to display the current one.
        # Filter by image name if possible, or just take tail(1)
        
        if len(df) > 0:
            row = df[df['image'] == img_path.name].iloc[-1]
            label = row['final_label']
            conf = row['final_confidence']
            
            # Display image + result
            img = cv2.imread(str(img_path))
            if img is not None:
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                plt.figure(figsize=(10, 6))
                plt.imshow(img)
                plt.title(f"{img_path.name}\nPrediction: {label.upper()} ({conf:.1%})", 
                          fontsize=14, 
                          color='green' if label!='none' else 'black')
                plt.axis('off')
                plt.show()
            
            print(f"Results for {img_path.name}:")
            
            # Robust DataFrame Construction
            # Identify patterns from probability columns
            p_cols = [c for c in df.columns if c.startswith('p_')]
            
            data_rows = []
            for p_c in p_cols:
                pat_name = p_c.replace('p_', '')
                t_c = f"thresh_{pat_name}"
                
                prob = row[p_c]
                
                if t_c in row:
                    thresh = row[t_c]
                    decision = 'YES' if prob >= thresh else 'NO'
                    data_rows.append({
                        'Pattern': pat_name,
                        'Probability': prob,
                        'Threshold': thresh,
                        'Decision': decision
                    })
                else:
                    # Fallback if threshold missing (shouldn't happen with new code)
                    data_rows.append({
                        'Pattern': pat_name,
                        'Probability': prob,
                        'Threshold': 'N/A',
                        'Decision': '?'
                    })
            
            if data_rows:
                disp_df = pd.DataFrame(data_rows)
                # Reorder columns
                disp_df = disp_df[['Pattern', 'Probability', 'Threshold', 'Decision']]
                print(disp_df.to_string(index=False))
            else:
                print("(No pattern scores found)")

            print("-" * 80)
            
    except Exception as e:
        print(f"❌ Error processing {img_path.name}: {e}")
        import traceback
        traceback.print_exc()

Starting Inference... (Plots will appear below)

                                     image final_label  final_confidence  p_head_and_shoulders  y_pred_head_and_shoulders  thresh_head_and_shoulders  p_double_top  y_pred_double_top  thresh_double_top  p_double_bottom  y_pred_double_bottom  thresh_double_bottom  p_ascending_triangle  y_pred_ascending_triangle  thresh_ascending_triangle
0  Screenshot from 2025-12-20 15-56-24.png  double_top          0.427924              0.543614                          0                        0.6      0.427924                  1                0.3         0.379881                     1                   0.3              0.243653                          0                       0.35
Results for Screenshot from 2025-12-20 15-56-24.png:
           Pattern  Probability  Threshold Decision
head_and_shoulders     0.543614       0.60       NO
        double_top     0.427924       0.30      YES
     double_bottom     0.379881       0.30      YES
ascending_tria